# Notebook 4 - Aandrijving, motor en optimalisatie

Deze notebook gebruikt belastingsresultaten uit een Notebook 3-case om de praktische aandrijving te dimensioneren. De mechanische belasting blijft de bron voor zwaartekracht, wrijving, aandrijfkracht, houdkracht en framebelasting. Notebook 4 vertaalt die resultaten naar een motor-, tandriem/kabel- en remconcept.

Workflow:

1. Run de kinematicanotebook.
2. Run de inertie-only dynamicanotebook.
3. Run `Notebook 3.ipynb` voor de baseline met zwaartekracht en wrijving.
4. Run eventueel `Notebook 3 - Trekveren.ipynb` voor de veer-case.
5. Kies bovenaan `load_case` en run deze notebook voor aandrijfkeuze, motorkoppel, vermogen, rem, precisie en ontwerpkeuzes.

De gekozen voorkeursarchitectuur is een 24 V DC/BLDC reductiemotor met encoder en rem, onderaan in de mastvoet. Die motor drijft een gesloten tandriem- of kabel/riemlus langs de mast aan. De riem levert alleen de verticale schuifkracht in de `s`-richting. De schuivergeleiding en de mast nemen de grote zijreacties op.



## Setup en data uit Notebook 3

Deze notebook herberekent de inverse dynamica niet. Ze leest een gekozen `.npz`-bestand uit een Notebook 3-case en gebruikt daaruit:

- `F_drive_s_total`: totale benodigde aandrijfkracht in positieve `s`-richting;
- `F_hold_s_curve`: statische houdkracht als functie van schuiverpositie;

Beschikbare standaard-loadcases zijn `baseline`, `trekveren`, `overdekking` en `overdekking_trekveren`. Met `custom` kan een los Notebook-3-compatibel `.npz`-bestand worden ingelezen.
- `ds`: schuiversnelheid; `s` is positief naar beneden;
- steunreacties aan schuiver en frame, om te bewaken dat de riem niet als geleiding wordt gebruikt.

De tekenconventie blijft dezelfde als in de vorige notebooks: openen betekent meestal `ds < 0`, en een negatieve `F_s` betekent dat de actuator omhoog moet trekken.



In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

load_case = "overdekking_trekveren"  # opties: "baseline", "trekveren", "overdekking", "overdekking_trekveren", "custom"
custom_results3_filename = "notebook3_gravity_friction_results.npz"

load_case_files = {
    "baseline": "notebook3_gravity_friction_results.npz",
    "trekveren": "notebook3_trekveren_results.npz",
    "overdekking": "notebook3_overdekking_results.npz",
    "overdekking_trekveren": "notebook3_overdekking_trekveren_results.npz",
}
if load_case == "custom":
    results3_path = Path(custom_results3_filename)
elif load_case in load_case_files:
    results3_path = Path(load_case_files[load_case])
else:
    raise ValueError("load_case moet 'baseline', 'trekveren', 'overdekking', 'overdekking_trekveren' of 'custom' zijn.")

if not results3_path.exists():
    if load_case == "trekveren":
        raise FileNotFoundError("Run eerst Notebook 3 - Trekveren.ipynb zodat notebook3_trekveren_results.npz bestaat.")
    if load_case in ("overdekking", "overdekking_trekveren"):
        raise FileNotFoundError("Run eerst Notebook 3 - Overdekking.ipynb zodat de overdekkingsresultaten bestaan.")
    raise FileNotFoundError(f"Kan belastingsbestand niet vinden: {results3_path}")

required_keys = [
    "t", "s", "ds", "dds",
    "F_drive_s_inertia", "F_drive_s_gravity", "F_drive_s_total",
    "F_gravity_component", "F_friction_component", "F_slider_drive_component", "F_pin_friction_component",
    "P_act_total", "E_act_total",
    "hold_s_curve", "F_hold_s_curve", "R_Ax_hold_curve", "static_slider_capacity_curve",
    "R_Ax_total", "C_x_total", "C_y_total", "F_A_total_norm", "F_C_total_norm", "F_frame_total_norm",
    "A_max_full", "P_avg_full", "P_peak_full", "P_rms_full", "F_peak_full", "F_rms_full",
    "total_model_mass", "total_weight", "g",
    "mu_slider", "mu_pin", "pin_radius", "dyn_residual_total", "max_diff_notebook2",
]

data = np.load(results3_path, allow_pickle=True)
missing = [key for key in required_keys if key not in data.files]
if missing:
    raise KeyError(f"Ontbrekende keys in {results3_path}: {missing}")


def cumulative_integral(time_values, y_values):
    out = np.zeros_like(y_values, dtype=float)
    out[1:] = np.cumsum(0.5 * (y_values[1:] + y_values[:-1]) * np.diff(time_values))
    return out


t = data["t"]; s = data["s"]; ds = data["ds"]; dds = data["dds"]
Ts = float(np.median(np.diff(t))) if len(t) > 1 else 0.0
T_run = float(t[-1] - t[0])

F_drive_s_inertia = data["F_drive_s_inertia"]
F_drive_s_gravity = data["F_drive_s_gravity"]
F_drive_s_total = data["F_drive_s_total"]
F_gravity_component = data["F_gravity_component"]
F_friction_component = data["F_friction_component"]
F_slider_drive_component = data["F_slider_drive_component"]
F_pin_friction_component = data["F_pin_friction_component"]
P_act_total = data["P_act_total"]
E_act_total = data["E_act_total"]

has_bidirectional = all(key in data.files for key in [
    "F_drive_s_total_open", "F_drive_s_total_close",
    "P_act_total_open", "P_act_total_close",
    "t_open", "t_close", "ds_open_profile", "ds_close_profile",
])

if has_bidirectional:
    direction_names = np.asarray(data["direction_names"]).astype(str)
    t_open_profile = data["t_open"]; s_open_profile = data["s_open_profile"]
    ds_open_profile = data["ds_open_profile"]; dds_open_profile = dds.copy()
    t_close_profile = data["t_close"]; s_close_profile = data["s_close_profile"]
    ds_close_profile = data["ds_close_profile"]; dds_close_profile = dds[::-1]
    F_drive_s_total_open = data["F_drive_s_total_open"]
    F_drive_s_total_close = data["F_drive_s_total_close"]
    F_drive_s_inertia_open = data["F_drive_s_inertia_open"] if "F_drive_s_inertia_open" in data.files else F_drive_s_inertia
    F_drive_s_inertia_close = data["F_drive_s_inertia_close"] if "F_drive_s_inertia_close" in data.files else F_drive_s_inertia[::-1]
    P_act_total_open = data["P_act_total_open"]
    P_act_total_close = data["P_act_total_close"]
    E_act_total_open = data["E_act_total_open"] if "E_act_total_open" in data.files else cumulative_integral(t_open_profile, P_act_total_open)
    E_act_total_close = data["E_act_total_close"] if "E_act_total_close" in data.files else cumulative_integral(t_close_profile, P_act_total_close)
else:
    direction_names = np.array(["openen"])
    t_open_profile = t.copy(); s_open_profile = s.copy(); ds_open_profile = ds.copy(); dds_open_profile = dds.copy()
    t_close_profile = None; s_close_profile = None; ds_close_profile = None; dds_close_profile = None
    F_drive_s_total_open = F_drive_s_total.copy(); F_drive_s_total_close = None
    F_drive_s_inertia_open = F_drive_s_inertia.copy(); F_drive_s_inertia_close = None
    P_act_total_open = P_act_total.copy(); P_act_total_close = None
    E_act_total_open = E_act_total.copy(); E_act_total_close = None

motion_mask_open = np.abs(ds_open_profile) > 1e-8
if not np.any(motion_mask_open):
    motion_mask_open = np.ones_like(ds_open_profile, dtype=bool)
if has_bidirectional:
    motion_mask_close = np.abs(ds_close_profile) > 1e-8
    if not np.any(motion_mask_close):
        motion_mask_close = np.ones_like(ds_close_profile, dtype=bool)
else:
    motion_mask_close = None

force_peak_open_one = float(np.max(np.abs(F_drive_s_total_open[motion_mask_open])))
power_peak_open_one = float(np.max(np.maximum(P_act_total_open, 0.0)))
energy_positive_open_one = float(data["E_positive_open"]) if "E_positive_open" in data.files else float(np.trapezoid(np.maximum(P_act_total_open, 0.0), t_open_profile))
energy_net_open_one = float(data["E_net_open"]) if "E_net_open" in data.files else float(np.trapezoid(P_act_total_open, t_open_profile))
if has_bidirectional:
    force_peak_close_one = float(np.max(np.abs(F_drive_s_total_close[motion_mask_close])))
    power_peak_close_one = float(np.max(np.maximum(P_act_total_close, 0.0)))
    energy_positive_close_one = float(data["E_positive_close"]) if "E_positive_close" in data.files else float(np.trapezoid(np.maximum(P_act_total_close, 0.0), t_close_profile))
    energy_net_close_one = float(data["E_net_close"]) if "E_net_close" in data.files else float(np.trapezoid(P_act_total_close, t_close_profile))
else:
    force_peak_close_one = np.nan; power_peak_close_one = np.nan
    energy_positive_close_one = np.nan; energy_net_close_one = np.nan

direction_force_peaks_one = np.array([force_peak_open_one] + ([] if not has_bidirectional else [force_peak_close_one]))
direction_power_peaks_one = np.array([power_peak_open_one] + ([] if not has_bidirectional else [power_peak_close_one]))
direction_positive_energy_one = np.array([energy_positive_open_one] + ([] if not has_bidirectional else [energy_positive_close_one]))
direction_net_energy_one = np.array([energy_net_open_one] + ([] if not has_bidirectional else [energy_net_close_one]))

design_direction_index = int(np.argmax(direction_force_peaks_one))
power_design_direction_index = int(np.argmax(direction_power_peaks_one))
motor_design_direction = direction_names[design_direction_index]
power_design_direction = direction_names[power_design_direction_index]
if has_bidirectional and design_direction_index == 1:
    t_drive = t_close_profile; s_drive_profile = s_close_profile; ds_drive = ds_close_profile; dds_drive = dds_close_profile
    F_s_one_design = F_drive_s_total_close; F_inertia_one_design = F_drive_s_inertia_close; P_one_design = P_act_total_close
else:
    t_drive = t_open_profile; s_drive_profile = s_open_profile; ds_drive = ds_open_profile; dds_drive = dds_open_profile
    F_s_one_design = F_drive_s_total_open; F_inertia_one_design = F_drive_s_inertia_open; P_one_design = P_act_total_open
Ts_drive = float(np.median(np.diff(t_drive))) if len(t_drive) > 1 else Ts
motion_mask_drive = np.abs(ds_drive) > 1e-8
if not np.any(motion_mask_drive):
    motion_mask_drive = np.ones_like(ds_drive, dtype=bool)

hold_s_curve = data["hold_s_curve"]
F_hold_s_curve = data["F_hold_s_curve"]
R_Ax_hold_curve = data["R_Ax_hold_curve"]
static_slider_capacity_curve = data["static_slider_capacity_curve"]

R_Ax_total = data["R_Ax_total"]
C_x_total = data["C_x_total"]
C_y_total = data["C_y_total"]
F_A_total_norm = data["F_A_total_norm"]
F_C_total_norm = data["F_C_total_norm"]
F_frame_total_norm = data["F_frame_total_norm"]

total_model_mass = float(data["total_model_mass"])
total_weight = float(data["total_weight"])
g = float(data["g"])

loadcase_mechanism_count = int(data["mechanism_count_total"]) if "mechanism_count_total" in data.files else None
canopy_width_loaded = float(data["canopy_width"]) if "canopy_width" in data.files else None
canopy_depth_loaded = float(data["canopy_depth"]) if "canopy_depth" in data.files else None
payload_mass_K_loaded = float(data["payload_mass_K_equivalent"]) if "payload_mass_K_equivalent" in data.files else (float(data["payload_mass_K"]) if "payload_mass_K" in data.files else None)
beam_I_loaded = float(data["beam_I"]) if "beam_I" in data.files else np.nan
beam_span_loaded = float(data["beam_span"]) if "beam_span" in data.files else np.nan
front_beam_mass_per_m_loaded = float(data["front_beam_mass_per_m"]) if "front_beam_mass_per_m" in data.files else np.nan
fabric_areal_density_loaded = float(data["fabric_areal_density"]) if "fabric_areal_density" in data.files else np.nan
front_beam_tributary_depth_fraction_loaded = float(data["front_beam_tributary_depth_fraction"]) if "front_beam_tributary_depth_fraction" in data.files else 0.5
aluminium_E_loaded = float(data["aluminium_E"]) if "aluminium_E" in data.files else np.nan

motion_mask = motion_mask_drive
stroke = float(np.max(s) - np.min(s))
move_time = float(t_open_profile[motion_mask_open][-1] - t_open_profile[motion_mask_open][0]) if np.any(motion_mask_open) else T_run
active_time = max(move_time, Ts)

print(f"Data ingeladen uit Notebook 3-case: {load_case}")
print(results3_path.resolve())
print(f"aantal tijdstappen                 : {len(t)}")
print(f"simulatieduur                      : {T_run:.3f} s")
print(f"effectieve bewegingstijd           : {active_time:.3f} s")
print(f"slag                               : {stroke:.4f} m")
print(f"totale bewegende modelmassa        : {total_model_mass:.3f} kg per mechanisme")
if loadcase_mechanism_count is not None:
    print(f"mechanismen in loadcase             : {loadcase_mechanism_count}")
    print(f"totale modelmassa loadcase          : {loadcase_mechanism_count * total_model_mass:.3f} kg")
if canopy_width_loaded is not None:
    print(f"overdekking breedte/uitval          : {canopy_width_loaded:.2f} m / {canopy_depth_loaded:.2f} m")
if payload_mass_K_loaded is not None:
    print(f"equivalente K-massa                 : {payload_mass_K_loaded:.3f} kg per mechanisme")
print(f"bidirectionele loadcase             : {has_bidirectional}")
for i, name in enumerate(direction_names):
    print(f"{name:8s} | max |F_s| = {direction_force_peaks_one[i]:.2f} N | max P+ = {direction_power_peaks_one[i]:.2f} W | E+ = {direction_positive_energy_one[i]:.2f} J")
print(f"maatgevende kracht-richting         : {motor_design_direction}")
print(f"maatgevende vermogens-richting      : {power_design_direction}")
print(f"max |F_hold|, per mechanisme        : {np.max(np.abs(F_hold_s_curve)):.3f} N")
validation_nb2 = float(data["max_diff_notebook2"])
if np.isfinite(validation_nb2):
    print(f"validatie t.o.v. Notebook 2         : {validation_nb2:.3e} N")
else:
    print("validatie t.o.v. Notebook 2         : n.v.t. voor eigen loadcase-massa")
if "inertia_check_residual" in data.files:
    print(f"interne inertiecheck residu          : {float(data['inertia_check_residual']):.3e}")
print(f"max dynamisch residu Notebook 3     : {np.max(data['dyn_residual_total']):.3e}")
if "dyn_residual_total_close" in data.files:
    print(f"max dynamisch residu sluiten         : {np.max(data['dyn_residual_total_close']):.3e}")
if "spring_count" in data.files:
    print(f"trekveerpakket                      : {int(data['spring_count'])} veren, F_open={float(data['spring_force_open_total']):.2f} N, F_closed={float(data['spring_force_closed_total']):.2f} N")


## Ontwerpkeuzes voor de aandrijving

De mechanische belasting wordt gesplitst in aandrijfkracht, houdkracht en steunreacties. Die splitsing bepaalt de aandrijfarchitectuur:

- de riem/kabel levert de verticale kracht langs `s`;
- de rem of vergrendeling houdt open, gesloten en tussenstanden vast;
- de schuivergeleiding neemt de zijreactie en het kantelmoment op;
- de controller, encoder en eindschakelaars zorgen voor herhaalbare positionering.

De notebook rekent daarom niet alleen een piekkracht uit, maar ook het nodige koppel, vermogen, remkoppel, poelietoerental, positiegevoeligheid en het effect van enkele ontwerpkeuzes.



In [ ]:
# ============================================================
# Instelbare aandrijfparameters
# ============================================================

drive_type = "belt_cable"       # praktische keuze: tandriem of kabel/riem langs de mast
mechanism_count_override = None  # None = gebruik mechanism_count_total uit loadcase indien aanwezig
mechanism_count = loadcase_mechanism_count if (mechanism_count_override is None and loadcase_mechanism_count is not None) else (1 if mechanism_count_override is None else int(mechanism_count_override))

# Poelie/trommel aan de motorreductor-uitgang.
# De notebook kan automatisch een radius kiezen uit de kandidaatset.
auto_select_pulley = True
preferred_pulley_radius = 0.025  # [m] voorkeur: 25 mm als die binnen de limieten blijft
pulley_radius_candidates = np.array([0.015, 0.020, 0.025, 0.030, 0.040, 0.050, 0.060])
minimum_practical_pulley_radius = 0.020  # [m] kleine poelies vragen meer aandacht voor riembuiging
max_preferred_drive_torque = 12.0  # [Nm] gewenste bovengrens voor aandrijfkoppel aan uitgang
max_preferred_brake_torque = 8.0   # [Nm] gewenste bovengrens voor remkoppel aan uitgang
manual_pulley_radius = 0.025      # [m] alleen gebruikt als auto_select_pulley = False

drive_efficiency = 0.65           # [-] globale efficientie tussen motor/reductor en riemkracht
gear_efficiency = 0.75            # [-] orde-grootte voor reductiekast

# Veiligheidsfactoren
drive_safety_factor = 2.0         # voor actieve kracht/koppel/vermogen
brake_safety_factor = 2.0         # voor stilstand en tussenstanden

# Praktische minimale ontwerpkracht voor riem/kabel.
# Deze vloer voorkomt dat een theoretisch kleine kracht tot een te lichte buitenconstructie leidt.
line_force_floor_single = 200.0   # [N]
line_force_floor_double = 300.0   # [N]
line_force_floor_multi_per_mechanism = 150.0  # [N] extra vloer voor meer dan twee mechanismen

# Motorconcept
motor_voltage = 24.0              # [V]
controller_current_margin = 1.50  # [-]
motor_nominal_speed_rpm = 3000.0  # [rpm] continu orde-grootte
motor_peak_speed_rpm = 6000.0     # [rpm] korte piek voor dit trage positioneersysteem

# Reductiekeuze: hoogste verhouding die de pieksnelheid nog toelaat geeft meer koppelreserve.
auto_select_gear_ratio = True
gear_ratio_candidates = np.array([25.0, 30.0, 40.0, 50.0, 60.0, 75.0, 100.0])
manual_gear_ratio = 60.0

# Gewenste uitgangssnelheid aan poelie/trommel.
target_output_rpm_min = 20.0      # [rpm]
target_output_rpm_max = 40.0      # [rpm]
allowable_peak_output_rpm = 120.0 # [rpm] korte piek aan poelie/reductoruitgang

# Precisie-inschatting
encoder_counts_per_motor_rev = 1024  # [counts/rev] eenvoudige encoder
encoder_decode_factor = 4            # quadrature decode
estimated_backlash_mm = 1.0          # [mm] conservatieve mechanische speling/elasticiteit
position_tolerance_mm = 5.0          # [mm] gewenste positioneernauwkeurigheid voor tussenstanden
effective_drive_stiffness = 2.0e5    # [N/m] riem + bevestiging + schuiver, ruwe orde-grootte

# Richtwaarden voor kost en energieverbruik
open_close_cycles_per_day = 1.0     # 1 cyclus = openen + sluiten
operating_days_per_year = 220.0
electricity_price_eur_per_kwh = 0.35
controller_standby_power_w = 0.0    # bewust 0: alleen bewegingsenergie van de aandrijving
motor_cost_min_eur = 350.0 if load_case.startswith("overdekking") else 120.0
motor_cost_max_eur = 900.0 if load_case.startswith("overdekking") else 350.0
drive_hardware_cost_min_eur = 250.0 if load_case.startswith("overdekking") else 100.0
drive_hardware_cost_max_eur = 700.0 if load_case.startswith("overdekking") else 300.0

# Motorclass-richtwaarden. Dit is geen cataloguskeuze, maar een controle op orde van grootte.
motor_class_names = np.array(["100 W DC gearmotor", "250 W DC/BLDC gearmotor", "500 W BLDC/servo + rem", "750 W servo + rem"])
motor_class_power_w = np.array([100.0, 250.0, 500.0, 750.0])
motor_class_output_torque_nm = np.array([10.0, 25.0, 50.0, 75.0])
motor_class_brake_torque_nm = np.array([0.0, 8.0, 25.0, 40.0])
motor_class_cost_min_eur = np.array([120.0, 250.0, 350.0, 600.0])
motor_class_cost_max_eur = np.array([250.0, 500.0, 900.0, 1400.0])
motor_class_has_encoder = np.array([False, True, True, True])
motor_class_has_brake = np.array([False, False, True, True])
selected_motor_ip_rating = "IP54/IP65 gewenst"

# Gemeenschappelijke aandrijfas boven/achter de constructie
shaft_options = [
    ("solid_30", 0.030, 0.000),
    ("solid_35", 0.035, 0.000),
    ("solid_40", 0.040, 0.000),
    ("tube_40x5", 0.040, 0.030),
    ("tube_50x5", 0.050, 0.040),
    ("tube_60x5", 0.060, 0.050),
]
shaft_material_G = 79e9              # [Pa] staal, orde-grootte
shaft_density = 7850.0               # [kg/m]
shaft_allowable_shear_stress = 80e6  # [Pa] conservatieve ontwerpwaarde
shaft_twist_limit_deg = 2.0          # [deg] over de volledige aslengte
shaft_torque_distribution_factor = 1.0  # 1.0 = motor aan een zijde, conservatief
belt_bearing_load_factor = 2.0        # riemspanning/lagerlast orde-grootte t.o.v. lokale lijnkracht

if drive_type != "belt_cable":
    raise ValueError("Deze notebook is uitgewerkt voor drive_type = 'belt_cable'.")
if mechanism_count < 1:
    raise ValueError("mechanism_count moet minstens 1 zijn.")
if drive_efficiency <= 0 or gear_efficiency <= 0:
    raise ValueError("Efficienties moeten positief zijn.")
if drive_safety_factor < 1 or brake_safety_factor < 1:
    raise ValueError("Veiligheidsfactoren moeten minstens 1 zijn.")
if np.any(pulley_radius_candidates <= 0):
    raise ValueError("Alle kandidaatpoelieradii moeten positief zijn.")
if np.any(gear_ratio_candidates <= 0):
    raise ValueError("Alle kandidaat-reducties moeten positief zijn.")

if mechanism_count == 1:
    line_force_floor = line_force_floor_single
elif mechanism_count == 2:
    line_force_floor = line_force_floor_double
else:
    line_force_floor = max(line_force_floor_double, line_force_floor_multi_per_mechanism * mechanism_count)

print("Aandrijfparameters:")
print(f"drive_type                         : {drive_type}")
print(f"mechanism_count                    : {mechanism_count}")
print(f"mechanism_count_override           : {mechanism_count_override}")
print(f"auto_select_pulley                 : {auto_select_pulley}")
print(f"preferred_pulley_radius            : {preferred_pulley_radius*1000:.1f} mm")
print(f"max_preferred_drive_torque         : {max_preferred_drive_torque:.1f} Nm")
print(f"drive_efficiency                   : {drive_efficiency:.2f}")
print(f"drive_safety_factor                : {drive_safety_factor:.2f}")
print(f"brake_safety_factor                : {brake_safety_factor:.2f}")
print(f"praktische vloer lijnkracht         : {line_force_floor:.1f} N")
print(f"motor_voltage                      : {motor_voltage:.1f} V")


## Kracht, vermogen en koppel

De riemkracht is de kracht die nodig is om de schuiver langs `s` te bewegen. Voor een symmetrisch dubbel mechanisme wordt de verticale aandrijfkracht ongeveer verdubbeld, omdat beide zijden tegelijk geopend worden.

Voor de poelie geldt:

$$
T_{poelie}(t) = \frac{F_s(t)\,r}{\eta}
$$

met `r` de poelieradius en `eta` de globale aandrijfefficientie. Het mechanische vermogen aan de schuiver is:

$$
P_s(t) = F_s(t)\,\dot{s}(t)
$$

Voor openen zijn `F_s` en `ds` meestal allebei negatief, waardoor het geleverde vermogen positief wordt.



In [ ]:
F_s_one = F_s_one_design
F_s_drive = mechanism_count * F_s_one
F_hold_drive_curve = mechanism_count * F_hold_s_curve

F_s_drive_open = mechanism_count * F_drive_s_total_open
P_slider_drive_open = F_s_drive_open * ds_open_profile
if has_bidirectional:
    F_s_drive_close = mechanism_count * F_drive_s_total_close
    P_slider_drive_close = F_s_drive_close * ds_close_profile
else:
    F_s_drive_close = None
    P_slider_drive_close = None

P_slider_drive = F_s_drive * ds_drive
P_positive_slider = np.maximum(P_slider_drive, 0.0)
P_negative_slider = np.minimum(P_slider_drive, 0.0)

line_force_peak_motion_open = mechanism_count * force_peak_open_one
line_force_peak_motion_close = mechanism_count * force_peak_close_one if has_bidirectional else 0.0
line_force_peak_motion = float(np.nanmax([line_force_peak_motion_open, line_force_peak_motion_close]))
line_force_peak_hold = float(np.max(np.abs(F_hold_drive_curve)))
line_force_peak_operating = max(line_force_peak_motion, line_force_peak_hold)
line_force_design_raw = drive_safety_factor * line_force_peak_operating
line_force_design = max(line_force_design_raw, line_force_floor)

avg_line_speed = stroke / active_time
peak_line_speed = float(max(np.max(np.abs(ds_open_profile[motion_mask_open])), np.max(np.abs(ds_close_profile[motion_mask_close])) if has_bidirectional else 0.0))


def pulley_metrics(radius):
    peak_rpm = peak_line_speed / (2.0 * np.pi * radius) * 60.0
    avg_rpm = avg_line_speed / (2.0 * np.pi * radius) * 60.0
    drive_torque = line_force_design * radius / drive_efficiency
    brake_torque = brake_safety_factor * line_force_peak_hold * radius
    mm_per_rev = 2.0 * np.pi * radius * 1000.0
    feasible = (
        (radius >= minimum_practical_pulley_radius)
        and (peak_rpm <= allowable_peak_output_rpm)
        and (drive_torque <= max_preferred_drive_torque)
        and (brake_torque <= max_preferred_brake_torque)
    )
    return avg_rpm, peak_rpm, drive_torque, brake_torque, mm_per_rev, feasible


radius_rows = []
for radius in pulley_radius_candidates:
    radius_rows.append((radius, *pulley_metrics(radius)))
radius_rows = np.array(radius_rows, dtype=float)

candidate_r = radius_rows[:, 0]
candidate_feasible = radius_rows[:, 6].astype(bool)
preferred_matches = np.isclose(candidate_r, preferred_pulley_radius)

if auto_select_pulley:
    if np.any(candidate_feasible & preferred_matches):
        selected_idx = int(np.where(candidate_feasible & preferred_matches)[0][0])
        pulley_selection_note = "voorkeursradius is haalbaar"
    elif np.any(candidate_feasible):
        feasible_indices = np.where(candidate_feasible)[0]
        selected_idx = int(feasible_indices[np.argmin(np.abs(candidate_r[feasible_indices] - preferred_pulley_radius))])
        pulley_selection_note = "voorkeursradius niet haalbaar; dichtstbij haalbare kandidaat gekozen"
    else:
        rpm_violation = np.maximum(radius_rows[:, 2] / allowable_peak_output_rpm - 1.0, 0.0)
        radius_violation = np.maximum(minimum_practical_pulley_radius / candidate_r - 1.0, 0.0)
        selected_idx = int(np.argmin(rpm_violation + radius_violation))
        pulley_selection_note = "geen kandidaat voldoet volledig; minst slechte kandidaat gekozen"
else:
    manual_matches = np.isclose(candidate_r, manual_pulley_radius)
    if np.any(manual_matches):
        selected_idx = int(np.where(manual_matches)[0][0])
    else:
        selected_idx = int(np.argmin(np.abs(candidate_r - manual_pulley_radius)))
    pulley_selection_note = "handmatige radius gebruikt"

drive_pulley_radius = float(radius_rows[selected_idx, 0])
avg_equiv_output_rpm = float(radius_rows[selected_idx, 1])
peak_output_rpm = float(radius_rows[selected_idx, 2])
pulley_radius_feasible = bool(radius_rows[selected_idx, 6])

pulley_speed_rps = ds_drive / (2.0 * np.pi * drive_pulley_radius)
pulley_speed_rpm = 60.0 * pulley_speed_rps
T_pulley_active = F_s_drive * drive_pulley_radius / drive_efficiency
T_pulley_peak_active = line_force_peak_motion * drive_pulley_radius / drive_efficiency
T_pulley_design = line_force_design * drive_pulley_radius / drive_efficiency

P_peak_slider_open = float(np.max(np.maximum(P_slider_drive_open, 0.0)))
P_rms_slider_open = float(np.sqrt(np.mean(P_slider_drive_open[motion_mask_open]**2)))
if has_bidirectional:
    P_peak_slider_close = float(np.max(np.maximum(P_slider_drive_close, 0.0)))
    P_rms_slider_close = float(np.sqrt(np.mean(P_slider_drive_close[motion_mask_close]**2)))
else:
    P_peak_slider_close = 0.0
    P_rms_slider_close = 0.0
P_peak_slider = max(P_peak_slider_open, P_peak_slider_close)
P_rms_slider = max(P_rms_slider_open, P_rms_slider_close)
P_peak_motor_input = drive_safety_factor * P_peak_slider / drive_efficiency
P_rms_motor_input = drive_safety_factor * P_rms_slider / drive_efficiency
I_peak_est = controller_current_margin * P_peak_motor_input / motor_voltage
I_rms_est = controller_current_margin * P_rms_motor_input / motor_voltage

E_positive_slider_open = mechanism_count * energy_positive_open_one
E_positive_slider_close = mechanism_count * energy_positive_close_one if has_bidirectional else 0.0
E_positive_slider = max(E_positive_slider_open, E_positive_slider_close)
E_net_slider_open = mechanism_count * energy_net_open_one
E_net_slider_close = mechanism_count * energy_net_close_one if has_bidirectional else 0.0
E_net_slider = E_net_slider_open if motor_design_direction == "openen" else E_net_slider_close

P_avg_drive = float(np.mean(P_slider_drive))
A_theta_drive = np.cumsum((P_slider_drive - P_avg_drive) * Ts_drive)
A_max_drive = float(np.max(A_theta_drive) - np.min(A_theta_drive))

if auto_select_gear_ratio:
    gear_feasible = gear_ratio_candidates * peak_output_rpm <= motor_peak_speed_rpm
    if np.any(gear_feasible):
        selected_gear_ratio = float(np.max(gear_ratio_candidates[gear_feasible]))
        gear_selection_note = "hoogste haalbare reductie binnen motorpieksnelheid"
    else:
        selected_gear_ratio = float(np.min(gear_ratio_candidates))
        gear_selection_note = "geen reductie voldoet volledig aan motorpieksnelheid"
else:
    selected_gear_ratio = float(manual_gear_ratio)
    gear_selection_note = "handmatige reductie gebruikt"

motor_speed_peak_est = peak_output_rpm * selected_gear_ratio
motor_speed_avg_est = avg_equiv_output_rpm * selected_gear_ratio
motor_torque_design_est = T_pulley_design / (selected_gear_ratio * gear_efficiency)

print("Aandrijfkracht en motorbelasting:")
print(f"piek lijnkracht per mechanisme, openen : {force_peak_open_one:.2f} N")
if has_bidirectional:
    print(f"piek lijnkracht per mechanisme, sluiten: {force_peak_close_one:.2f} N")
print(f"piek lijnkracht totaal beweging        : {line_force_peak_motion:.2f} N")
print(f"piek houdkracht totaal alle standen    : {line_force_peak_hold:.2f} N")
print(f"ontwerpkracht zonder praktische vloer  : {line_force_design_raw:.2f} N")
print(f"gekozen ontwerplijnkracht              : {line_force_design:.2f} N")
print()
print("Poelie- en reductiekeuze:")
print(f"gekozen poelieradius                   : {drive_pulley_radius*1000:.1f} mm ({pulley_selection_note})")
print(f"gemiddeld uitgangstoerental            : {avg_equiv_output_rpm:.1f} rpm")
print(f"piek-uitgangstoerental                 : {peak_output_rpm:.1f} rpm")
print(f"gekozen reductie                       : {selected_gear_ratio:.0f}:1 ({gear_selection_note})")
print(f"geschatte motorpieksnelheid            : {motor_speed_peak_est:.0f} rpm")
print(f"geschatte gemiddelde motorsnelheid     : {motor_speed_avg_est:.0f} rpm")
print()
print(f"piek aandrijfkoppel incl. verliezen    : {T_pulley_peak_active:.3f} Nm")
print(f"ontwerp aandrijfkoppel incl. verlies   : {T_pulley_design:.3f} Nm")
print(f"geschat motorkoppel voor reductie      : {motor_torque_design_est:.3f} Nm")
print(f"piek positief schuiververmogen openen  : {P_peak_slider_open:.2f} W")
if has_bidirectional:
    print(f"piek positief schuiververmogen sluiten : {P_peak_slider_close:.2f} W")
print(f"piek motor-ingangsvermogen ontwerp     : {P_peak_motor_input:.2f} W")
print(f"RMS motor-ingangsvermogen ontwerp      : {P_rms_motor_input:.2f} W")
print(f"geschatte piekstroom bij {motor_voltage:.0f} V          : {I_peak_est:.2f} A")
print(f"geschatte RMS-stroom bij {motor_voltage:.0f} V           : {I_rms_est:.2f} A")
print()
print(f"positieve mechanische arbeid openen    : {E_positive_slider_open:.2f} J")
if has_bidirectional:
    print(f"positieve mechanische arbeid sluiten   : {E_positive_slider_close:.2f} J")
print(f"arbeids-surplus aandrijving            : {A_max_drive:.2f} J")

if not pulley_radius_feasible:
    print("Waarschuwing: de gekozen poelieradius voldoet niet volledig aan de ingestelde grenzen.")
if motor_speed_peak_est > motor_peak_speed_rpm:
    print("Waarschuwing: de gekozen reductie vraagt meer motorpieksnelheid dan ingesteld.")

fig_load, ax_load = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
fig_load.suptitle("Belasting van de riem-/motoraandrijving")

ax_load[0, 0].plot(t_open_profile, F_s_drive_open, label="openen")
if has_bidirectional:
    ax_load[0, 0].plot(t_close_profile, F_s_drive_close, "--", label="sluiten")
ax_load[0, 0].axhline(line_force_design, color="tab:red", ls="--", label="ontwerpkracht")
ax_load[0, 0].axhline(-line_force_design, color="tab:red", ls="--")
ax_load[0, 0].set_title("Lijnkracht in de aandrijving")
ax_load[0, 0].set_xlabel("t [s]"); ax_load[0, 0].set_ylabel("N")
ax_load[0, 0].grid(True); ax_load[0, 0].legend()

ax_load[0, 1].plot(t_drive, T_pulley_active, label=f"T actief ({motor_design_direction})")
ax_load[0, 1].axhline(T_pulley_design, color="tab:red", ls="--", label="ontwerpkoppel")
ax_load[0, 1].axhline(-T_pulley_design, color="tab:red", ls="--")
ax_load[0, 1].set_title("Aandrijfkoppel inclusief verliezen")
ax_load[0, 1].set_xlabel("t [s]"); ax_load[0, 1].set_ylabel("Nm")
ax_load[0, 1].grid(True); ax_load[0, 1].legend()

ax_load[1, 0].plot(t_open_profile, P_slider_drive_open, label="openen")
if has_bidirectional:
    ax_load[1, 0].plot(t_close_profile, P_slider_drive_close, "--", label="sluiten")
ax_load[1, 0].axhline(0.0, color="black", lw=0.8)
ax_load[1, 0].set_title("Mechanisch vermogen aan de schuiver")
ax_load[1, 0].set_xlabel("t [s]"); ax_load[1, 0].set_ylabel("W")
ax_load[1, 0].grid(True); ax_load[1, 0].legend()

ax_load[1, 1].plot(t_drive, pulley_speed_rpm, label=f"poelie ({motor_design_direction})")
ax_load[1, 1].axhline(target_output_rpm_max, color="tab:orange", ls="--", label="richtwaarde continu")
ax_load[1, 1].axhline(-target_output_rpm_max, color="tab:orange", ls="--")
ax_load[1, 1].axhline(allowable_peak_output_rpm, color="tab:red", ls=":", label="toelaatbare piek")
ax_load[1, 1].axhline(-allowable_peak_output_rpm, color="tab:red", ls=":")
ax_load[1, 1].set_title("Poelietoerental")
ax_load[1, 1].set_xlabel("t [s]"); ax_load[1, 1].set_ylabel("rpm")
ax_load[1, 1].grid(True); ax_load[1, 1].legend()

plt.show()


## Houdkracht, rem en tussenstanden

Een riem houdt de overdekking niet vanzelf vast. De tussenstand wordt vastgehouden door de motorregeling, een zelfremmende overbrenging of beter: een echte rem/vergrendeling.

Voor de rem op de poelie-/reductoruitgang wordt gerekend met:

$$
T_{rem} \ge SF_{rem}\, |F_{hold}|\, r
$$

Er wordt bewust niet gerekend op toevallige schuiverwrijving. Die wrijving kan veranderen door slijtage, water, vuil, temperatuur of smering.



In [ ]:
T_hold_output_curve = np.abs(F_hold_drive_curve) * drive_pulley_radius
T_hold_brake_design_curve = brake_safety_factor * T_hold_output_curve
T_hold_motor_shaft_curve = T_hold_brake_design_curve / (selected_gear_ratio * gear_efficiency)
T_hold_motor_conservative_curve = brake_safety_factor * np.abs(F_hold_drive_curve) * drive_pulley_radius / drive_efficiency

half_s = 0.5 * (np.min(hold_s_curve) + np.max(hold_s_curve))
i_open = int(np.argmin(hold_s_curve))
i_closed = int(np.argmax(hold_s_curve))
i_half = int(np.argmin(np.abs(hold_s_curve - half_s)))
i_hold_peak = int(np.argmax(np.abs(F_hold_drive_curve)))

print("Houdanalyse voor tussenstanden:")
for label, idx in [("open", i_open), ("halfopen", i_half), ("gesloten", i_closed), ("max", i_hold_peak)]:
    print(
        f"{label:9s} | s = {hold_s_curve[idx]:.3f} m | "
        f"|F_hold| = {abs(F_hold_drive_curve[idx]):.2f} N | "
        f"T_rem uitgang = {T_hold_brake_design_curve[idx]:.3f} Nm | "
        f"T_rem motoras ~ {T_hold_motor_shaft_curve[idx]:.3f} Nm"
    )

print()
print(f"max remkoppel aan poelie-/reductoruitgang : {np.max(T_hold_brake_design_curve):.3f} Nm")
print(f"max remkoppel omgerekend naar motoras     : {np.max(T_hold_motor_shaft_curve):.3f} Nm")
print(f"conservatief houdkoppel incl. verliezen   : {np.max(T_hold_motor_conservative_curve):.3f} Nm")
print(f"max theoretische schuiverwrijvingsgrens   : {np.max(static_slider_capacity_curve)*mechanism_count:.2f} N")
print("Conclusie: gebruik een rem/vergrendeling; ontwerp niet op schuiverwrijving alleen.")

fig_hold, ax_hold = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
fig_hold.suptitle("Houdkracht en remdimensionering")

ax_hold[0].plot(hold_s_curve, F_hold_drive_curve, label="F_hold")
ax_hold[0].axhline(0.0, color="black", lw=0.8)
ax_hold[0].set_title("Statische houdkracht")
ax_hold[0].set_xlabel("s [m]"); ax_hold[0].set_ylabel("N")
ax_hold[0].grid(True); ax_hold[0].legend()

ax_hold[1].plot(hold_s_curve, T_hold_brake_design_curve, label="uitgang")
ax_hold[1].plot(hold_s_curve, T_hold_motor_shaft_curve, "--", label="motoras, benaderd")
ax_hold[1].set_title("Benodigd remkoppel")
ax_hold[1].set_xlabel("s [m]"); ax_hold[1].set_ylabel("Nm")
ax_hold[1].grid(True); ax_hold[1].legend()

plt.show()


## Poelieradius en motorselectie

De poelieradius geeft een duidelijke afweging:

- kleinere radius: minder koppel nodig, maar hoger toerental en meer riembuiging;
- grotere radius: lager toerental en meer verplaatsing per omwenteling, maar meer koppel nodig;
- voor precisie geeft een kleinere radius een kleinere verplaatsing per omwenteling, maar de echte nauwkeurigheid wordt meestal beperkt door speling, riemrek en schuivergeleiding.

De notebook kiest automatisch een radius uit `pulley_radius_candidates`. De voorkeur blijft 25 mm als die binnen de ingestelde grenzen past. Als het traject of de geometrie wijzigt en die radius niet meer haalbaar is, kiest de notebook de dichtstbijzijnde haalbare kandidaat.



In [ ]:
print("Invloed van poelieradius:")
print("r [mm] | avg rpm | peak rpm | T_drive incl. verlies [Nm] | T_rem uitgang [Nm] | mm/rev | status")
for radius, rpm_avg, rpm_peak, T_design, T_hold_design, mm_per_rev, feasible in radius_rows:
    status = "gekozen" if np.isclose(radius, drive_pulley_radius) else ("haalbaar" if feasible else "niet haalbaar")
    print(
        f"{1000*radius:6.1f} | {rpm_avg:7.1f} | {rpm_peak:8.1f} | "
        f"{T_design:26.2f} | {T_hold_design:16.2f} | {mm_per_rev:6.1f} | {status}"
    )

gear_rows = []
for ratio in gear_ratio_candidates:
    motor_peak = ratio * peak_output_rpm
    motor_avg = ratio * avg_equiv_output_rpm
    motor_torque = T_pulley_design / (ratio * gear_efficiency)
    feasible = motor_peak <= motor_peak_speed_rpm
    gear_rows.append((ratio, motor_avg, motor_peak, motor_torque, feasible))
gear_rows = np.array(gear_rows, dtype=float)

print()
print("Invloed van reductieverhouding:")
print("ratio | motor avg rpm | motor peak rpm | motor torque ontwerp [Nm] | status")
for ratio, motor_avg, motor_peak, motor_torque, feasible in gear_rows:
    status = "gekozen" if np.isclose(ratio, selected_gear_ratio) else ("haalbaar" if feasible else "te snel")
    print(f"{ratio:5.0f} | {motor_avg:13.0f} | {motor_peak:14.0f} | {motor_torque:24.3f} | {status}")

fig_radius, ax_radius = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
fig_radius.suptitle("Poelie- en reductiekeuze")

radius_mm = radius_rows[:, 0] * 1000.0
selected_radius_mm = drive_pulley_radius * 1000.0

ax_radius[0].plot(radius_mm, radius_rows[:, 2], "o-", label="piek rpm")
ax_radius[0].axhline(allowable_peak_output_rpm, color="tab:red", ls="--", label="limiet")
ax_radius[0].axvline(selected_radius_mm, color="black", ls=":", label="gekozen")
ax_radius[0].set_xlabel("r [mm]"); ax_radius[0].set_ylabel("rpm")
ax_radius[0].set_title("Poelietoerental")
ax_radius[0].grid(True); ax_radius[0].legend()

ax_radius[1].plot(radius_mm, radius_rows[:, 3], "o-", label="aandrijfkoppel")
ax_radius[1].plot(radius_mm, radius_rows[:, 4], "s--", label="remkoppel")
ax_radius[1].axvline(selected_radius_mm, color="black", ls=":", label="gekozen")
ax_radius[1].set_xlabel("r [mm]"); ax_radius[1].set_ylabel("Nm")
ax_radius[1].set_title("Koppel")
ax_radius[1].grid(True); ax_radius[1].legend()

ax_radius[2].plot(gear_rows[:, 0], gear_rows[:, 2], "o-", label="motor piek rpm")
ax_radius[2].axhline(motor_peak_speed_rpm, color="tab:red", ls="--", label="motor piekgrens")
ax_radius[2].axvline(selected_gear_ratio, color="black", ls=":", label="gekozen")
ax_radius[2].set_xlabel("reductie i [-]"); ax_radius[2].set_ylabel("rpm")
ax_radius[2].set_title("Reductie en motorsnelheid")
ax_radius[2].grid(True); ax_radius[2].legend()

plt.show()


## Motorclass en kostmarge

De berekende belasting wordt hier vertaald naar een realistische motorcategorie. De tabel is geen productselectie, maar voorkomt dat de kost en het vermogen los staan van de berekende koppel-, rem- en vermogenseis.


In [ ]:
recommended_motor_power_floor = 50.0 if mechanism_count == 1 else max(100.0, 50.0 * mechanism_count)
recommended_motor_power_peak = max(recommended_motor_power_floor, 1.5 * P_peak_motor_input)
recommended_output_torque = max(T_pulley_design, 8.0 if mechanism_count == 1 else max(12.0, 6.0 * mechanism_count))
recommended_brake_torque = max(np.max(T_hold_brake_design_curve), 2.0)

motor_class_suitable = (
    (motor_class_power_w >= recommended_motor_power_peak)
    & (motor_class_output_torque_nm >= recommended_output_torque)
    & (motor_class_brake_torque_nm >= recommended_brake_torque)
    & motor_class_has_encoder
    & motor_class_has_brake
)
if np.any(motor_class_suitable):
    selected_motor_class_index = int(np.where(motor_class_suitable)[0][0])
else:
    combined_margin = np.minimum.reduce([
        motor_class_power_w / recommended_motor_power_peak,
        motor_class_output_torque_nm / recommended_output_torque,
        np.maximum(motor_class_brake_torque_nm, 1e-9) / recommended_brake_torque,
    ])
    selected_motor_class_index = int(np.argmax(combined_margin))

selected_motor_class_name = str(motor_class_names[selected_motor_class_index])
selected_motor_power_class = float(motor_class_power_w[selected_motor_class_index])
selected_output_torque_class = float(motor_class_output_torque_nm[selected_motor_class_index])
selected_brake_torque_class = float(motor_class_brake_torque_nm[selected_motor_class_index])
selected_motor_cost_min = float(motor_class_cost_min_eur[selected_motor_class_index])
selected_motor_cost_max = float(motor_class_cost_max_eur[selected_motor_class_index])
motor_power_margin = selected_motor_power_class / recommended_motor_power_peak
motor_torque_margin = selected_output_torque_class / recommended_output_torque
motor_brake_margin = selected_brake_torque_class / recommended_brake_torque if recommended_brake_torque > 0 else np.inf

print("Motorclass-controle:")
print("klasse | P [W] | T_out [Nm] | T_rem [Nm] | encoder | rem | status")
for i, name in enumerate(motor_class_names):
    status = "gekozen" if i == selected_motor_class_index else ("geschikt" if motor_class_suitable[i] else "niet genoeg marge")
    print(
        f"{name:24s} | {motor_class_power_w[i]:6.0f} | {motor_class_output_torque_nm[i]:10.1f} | "
        f"{motor_class_brake_torque_nm[i]:10.1f} | {str(bool(motor_class_has_encoder[i])):7s} | {str(bool(motor_class_has_brake[i])):5s} | {status}"
    )
print()
print(f"vereist praktisch vermogen       : {recommended_motor_power_peak:.1f} W")
print(f"vereist uitgangskoppel           : {recommended_output_torque:.1f} Nm")
print(f"vereist remkoppel                : {recommended_brake_torque:.1f} Nm")
print(f"gekozen motorclass               : {selected_motor_class_name}")
print(f"marges P/T/rem                   : {motor_power_margin:.2f} / {motor_torque_margin:.2f} / {motor_brake_margin:.2f}")
print(f"gewenste bescherming             : {selected_motor_ip_rating}")

fig_motor_class, ax_motor_class = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
fig_motor_class.suptitle("Motorclass versus ontwerpvereiste")
indices = np.arange(len(motor_class_names))
colors = ["tab:green" if ok else "0.7" for ok in motor_class_suitable]
colors[selected_motor_class_index] = "tab:blue"
ax_motor_class[0].bar(indices, motor_class_power_w, color=colors)
ax_motor_class[0].axhline(recommended_motor_power_peak, color="tab:red", ls="--", label="vereist")
ax_motor_class[0].set_xticks(indices); ax_motor_class[0].set_xticklabels(["100", "250", "500", "750"])
ax_motor_class[0].set_xlabel("motorclass [W]"); ax_motor_class[0].set_ylabel("W")
ax_motor_class[0].set_title("Vermogen")
ax_motor_class[0].grid(True, axis="y"); ax_motor_class[0].legend()

ax_motor_class[1].bar(indices - 0.18, motor_class_output_torque_nm, width=0.35, label="uitgangskoppel", color=colors)
ax_motor_class[1].bar(indices + 0.18, motor_class_brake_torque_nm, width=0.35, label="remkoppel", color="tab:orange")
ax_motor_class[1].axhline(recommended_output_torque, color="tab:red", ls="--", label="T vereist")
ax_motor_class[1].axhline(recommended_brake_torque, color="tab:purple", ls=":", label="rem vereist")
ax_motor_class[1].set_xticks(indices); ax_motor_class[1].set_xticklabels(["100", "250", "500", "750"])
ax_motor_class[1].set_xlabel("motorclass [W]"); ax_motor_class[1].set_ylabel("Nm")
ax_motor_class[1].set_title("Koppel en rem")
ax_motor_class[1].grid(True, axis="y"); ax_motor_class[1].legend()
plt.show()


## Precisie van de aandrijving

Voor precisie zijn drie niveaus belangrijk:

1. **Meetresolutie:** encoder op motor of uitgangsas.
2. **Transmissie:** riemspanning, reductiespeling en poelieradius.
3. **Mechanische structuur:** stijve schuivergeleiding, zodat de zijreactie geen kanteling of blokkering veroorzaakt.

De theoretische encoderresolutie is meestal veel kleiner dan de echte mechanische fout. Daarom is een stijve, spelingsarme schuiver/collar belangrijker dan alleen een hogere encoderresolutie.



In [ ]:
line_per_output_rev = 2.0 * np.pi * drive_pulley_radius
encoder_counts_effective = encoder_counts_per_motor_rev * encoder_decode_factor
linear_resolution_motor_encoder = line_per_output_rev / (selected_gear_ratio * encoder_counts_effective)
linear_resolution_output_encoder = line_per_output_rev / encoder_counts_effective

elastic_deflection_peak = line_force_peak_operating / effective_drive_stiffness
elastic_deflection_design = line_force_design / effective_drive_stiffness

print("Precisie-inschatting:")
print(f"verplaatsing per poelieomwenteling        : {line_per_output_rev*1000:.2f} mm/rev")
print(f"resolutie motorencoder na reductie        : {linear_resolution_motor_encoder*1000:.4f} mm/count")
print(f"resolutie encoder direct op uitgang        : {linear_resolution_output_encoder*1000:.4f} mm/count")
print(f"geschatte mechanische speling/elasticiteit : {estimated_backlash_mm:.2f} mm")
print(f"elastische verplaatsing bij piekbelasting  : {elastic_deflection_peak*1000:.3f} mm")
print(f"elastische verplaatsing bij ontwerpkracht  : {elastic_deflection_design*1000:.3f} mm")
print(f"gewenste positioneernauwkeurigheid         : {position_tolerance_mm:.2f} mm")

precision_ok = estimated_backlash_mm + elastic_deflection_peak * 1000.0 <= position_tolerance_mm
if precision_ok:
    print("Conclusie: de orde-grootte is voldoende voor overdekkingspositionering, mits de schuivergeleiding spelingsarm is.")
else:
    print("Conclusie: de mechanische speling/stijfheid wordt kritischer dan encoderresolutie.")

fig_precision, ax_precision = plt.subplots(1, 1, figsize=(8, 4), constrained_layout=True)
fig_precision.suptitle("Precisiebronnen")

labels = ["motorencoder", "uitgangsencoder", "speling", "riemrek piek", "riemrek ontwerp"]
values_mm = [
    linear_resolution_motor_encoder * 1000.0,
    linear_resolution_output_encoder * 1000.0,
    estimated_backlash_mm,
    elastic_deflection_peak * 1000.0,
    elastic_deflection_design * 1000.0,
]
ax_precision.bar(labels, values_mm)
ax_precision.axhline(position_tolerance_mm, color="tab:red", ls="--", label="tolerantie")
ax_precision.set_ylabel("mm")
ax_precision.grid(True, axis="y")
ax_precision.legend()
plt.xticks(rotation=20)
plt.show()


## Energieverbruik en trajectkeuze

Een bewegingswet met hogere versnelling of ruk vraagt meer dynamische belasting. Voor deze overdekking is de inertiecomponent echter klein ten opzichte van de positie-afhankelijke belasting: zwaartekracht, wrijving en eventueel trekveerassistentie. Daardoor verlaagt trager bewegen vooral het piekvermogen en de rustiger regeling, maar niet de zwaartekrachtarbeid zelf.

De volgende cel maakt een eenvoudige schaalanalyse: hetzelfde pad wordt sneller of trager afgelegd. De inertiecomponent wordt benaderd als evenredig met `1/tijd_schaal^2`; zwaartekracht, Coulombachtige wrijving en eventuele veerassistentie blijven ongeveer dezelfde krachtcomponent. Dit is geen vervanging van Notebook 3, maar een compacte ontwerpcheck.



In [ ]:
time_scale_factors = np.array([0.60, 0.80, 1.00, 1.25, 1.50, 2.00])
non_inertial_component = F_s_drive - mechanism_count * F_inertia_one_design

scale_rows = []
for lam in time_scale_factors:
    # lam > 1 betekent trager; lam < 1 betekent sneller.
    F_scaled = non_inertial_component + mechanism_count * F_inertia_one_design / lam**2
    ds_scaled = ds_drive / lam
    t_scaled = (t_drive - t_drive[0]) * lam + t_drive[0]
    P_scaled = F_scaled * ds_scaled
    E_pos_scaled = np.trapezoid(np.maximum(P_scaled, 0.0), t_scaled)
    P_peak_scaled = np.max(np.maximum(P_scaled[motion_mask_drive], 0.0))
    F_peak_scaled = np.max(np.abs(F_scaled[motion_mask_drive]))
    A_theta_scaled = np.cumsum((P_scaled - np.mean(P_scaled)) * Ts_drive * lam)
    A_max_scaled = np.max(A_theta_scaled) - np.min(A_theta_scaled)
    scale_rows.append((lam, active_time * lam, F_peak_scaled, P_peak_scaled, E_pos_scaled, A_max_scaled))

print("Trajectschaal-trade-off:")
print("factor | beweegtijd [s] | peak |F_s| [N] | peak P [W] | positieve arbeid [J] | A_max [J]")
for lam, T_move_scaled, F_peak_scaled, P_peak_scaled, E_pos_scaled, A_max_scaled in scale_rows:
    print(f"{lam:5.2f} | {T_move_scaled:13.2f} | {F_peak_scaled:12.2f} | {P_peak_scaled:10.2f} | {E_pos_scaled:18.2f} | {A_max_scaled:8.2f}")

fig_scale, ax_scale = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
fig_scale.suptitle(f"Effect van sneller/trager traject ({motor_design_direction})")

ax_scale[0].plot(time_scale_factors, [row[2] for row in scale_rows], "o-")
ax_scale[0].set_xlabel("tijd_schaal [-]"); ax_scale[0].set_ylabel("N")
ax_scale[0].set_title("Piek kracht")
ax_scale[0].grid(True)

ax_scale[1].plot(time_scale_factors, [row[3] for row in scale_rows], "o-")
ax_scale[1].set_xlabel("tijd_schaal [-]"); ax_scale[1].set_ylabel("W")
ax_scale[1].set_title("Piek vermogen")
ax_scale[1].grid(True)

ax_scale[2].plot(time_scale_factors, [row[4] for row in scale_rows], "o-", label="positieve arbeid")
ax_scale[2].plot(time_scale_factors, [row[5] for row in scale_rows], "s--", label="A_max")
ax_scale[2].set_xlabel("tijd_schaal [-]"); ax_scale[2].set_ylabel("J")
ax_scale[2].set_title("Arbeid en arbeids-surplus")
ax_scale[2].grid(True); ax_scale[2].legend()

plt.show()


## Krachtgeneratie en symmetrische uitvoering

Voor meer kracht kan men een sterkere motor, grotere reductie of kleinere poelie kiezen. Maar de motor is niet de enige ontwerpfactor. De grote zijreactie aan de schuiver moet door de geleiding en mast opgenomen worden, niet door de riem.

Een symmetrische dubbele uitvoering rond de mast is mechanisch aantrekkelijk: de verticale aandrijfkracht wordt ongeveer dubbel, maar horizontale reacties kunnen elkaar grotendeels opheffen als de geometrie echt gespiegeld en stijf verbonden is. Dat verlaagt vooral lokale mastbelasting en kanteling, niet de benodigde verticale arbeid.



In [ ]:
counts = np.unique(np.array([1, 2, mechanism_count], dtype=int))
comparison_rows = []
for count in counts:
    floor = line_force_floor_single if count == 1 else (line_force_floor_double if count == 2 else max(line_force_floor_double, line_force_floor_multi_per_mechanism * count))
    F_peak_motion_count = count * float(np.nanmax(direction_force_peaks_one))
    F_peak_hold_count = count * np.max(np.abs(F_hold_s_curve))
    F_peak_operating_count = max(F_peak_motion_count, F_peak_hold_count)
    F_design_count = max(drive_safety_factor * F_peak_operating_count, floor)
    T_design_count = F_design_count * drive_pulley_radius / drive_efficiency
    P_peak_count = count * float(np.nanmax(direction_power_peaks_one))
    P_input_design_count = drive_safety_factor * P_peak_count / drive_efficiency
    comparison_rows.append((count, F_peak_operating_count, F_design_count, T_design_count, P_input_design_count))

# Lokale zijreactie uit Notebook 3 voor een enkel mechanisme.
side_reaction_peak = float(np.max(np.abs(R_Ax_total)))
local_bending_est = float(np.max(np.abs(R_Ax_total * s)))

print("Vergelijking aantal parallel aangedreven mechanismen:")
print("count | peak operation force [N] | design line force [N] | design torque [Nm] | design peak input P [W]")
for row in comparison_rows:
    print(f"{row[0]:5d} | {row[1]:24.2f} | {row[2]:21.2f} | {row[3]:18.2f} | {row[4]:21.2f}")

print()
print("Geleiding en mastbelasting:")
print(f"max lokale zijreactie schuiver A_x, enkel mechanisme : {side_reaction_peak:.2f} N")
print(f"ruwe momentarm-check max |A_x * s|                 : {local_bending_est:.2f} Nm")
print("Bij twee mechanismen op 6 m breedte kan de globale horizontale resultante kleiner worden,")
print("maar elke schuiver/geleider moet lokaal nog steeds zijn eigen zijreactie dragen.")

fig_count, ax_count = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
fig_count.suptitle("Parallel aangedreven mechanismen")

ax_count[0].bar([str(row[0]) for row in comparison_rows], [row[2] for row in comparison_rows])
ax_count[0].set_xlabel("aantal mechanismen")
ax_count[0].set_ylabel("N")
ax_count[0].set_title("Ontwerplijnkracht")
ax_count[0].grid(True, axis="y")

ax_count[1].bar([str(row[0]) for row in comparison_rows], [row[3] for row in comparison_rows])
ax_count[1].set_xlabel("aantal mechanismen")
ax_count[1].set_ylabel("Nm")
ax_count[1].set_title("Ontwerpkoppel")
ax_count[1].grid(True, axis="y")

plt.show()


## Gemeenschappelijke aandrijfas en synchronisatie

Bij meerdere mechanismen kan een motor gebruikt worden als de beweging mechanisch gesynchroniseerd wordt via een gemeenschappelijke as. De as hoort boven of achter de constructie, niet op wandelhoogte. Deze controle bekijkt of de orde van grootte van het askoppel, de torsiespanning en de verdraaiing praktisch verdedigbaar zijn.


In [ ]:
# ============================================================
# Gemeenschappelijke aandrijfas en synchronisatie
# ============================================================

shaft_span = float(canopy_width_loaded) if canopy_width_loaded is not None and canopy_width_loaded > 0 else max(1.0, float(mechanism_count - 1))
shaft_design_torque = float(T_pulley_design * shaft_torque_distribution_factor)
shaft_peak_active_torque = float(abs(T_pulley_peak_active) * shaft_torque_distribution_factor)
shaft_torque_per_slider_design = float(T_pulley_design / mechanism_count) if mechanism_count > 0 else np.nan
shaft_twist_limit_rad = np.deg2rad(shaft_twist_limit_deg)
local_line_force_design = line_force_design / mechanism_count
shaft_radial_bearing_load_est = belt_bearing_load_factor * local_line_force_design
shaft_coupling_torque_design = shaft_design_torque

shaft_rows = []
for name, outer_diameter, inner_diameter in shaft_options:
    J_polar = np.pi * (outer_diameter**4 - inner_diameter**4) / 32.0
    area = np.pi * (outer_diameter**2 - inner_diameter**2) / 4.0
    mass_per_m = area * shaft_density
    tau_max = shaft_design_torque * (0.5 * outer_diameter) / J_polar
    twist_rad = shaft_design_torque * shaft_span / (shaft_material_G * J_polar)
    util_shear = tau_max / shaft_allowable_shear_stress
    util_twist = abs(twist_rad) / shaft_twist_limit_rad
    ok = (util_shear <= 1.0) and (util_twist <= 1.0)
    shaft_rows.append((outer_diameter, inner_diameter, J_polar, mass_per_m, tau_max, twist_rad, util_shear, util_twist, ok))
shaft_rows = np.array(shaft_rows, dtype=float)
shaft_option_names = np.array([row[0] for row in shaft_options], dtype=str)

shaft_ok_mask = shaft_rows[:, 8].astype(bool)
if np.any(shaft_ok_mask):
    ok_indices = np.where(shaft_ok_mask)[0]
    shaft_selected_index = int(ok_indices[np.argmin(shaft_rows[ok_indices, 3])])
else:
    shaft_selected_index = int(np.argmin(np.maximum(shaft_rows[:, 6], shaft_rows[:, 7])))
shaft_selected_name = str(shaft_option_names[shaft_selected_index])
shaft_selected_diameter = float(shaft_rows[shaft_selected_index, 0])
shaft_selected_inner_diameter = float(shaft_rows[shaft_selected_index, 1])
shaft_selected_mass_per_m = float(shaft_rows[shaft_selected_index, 3])
shaft_selected_twist_deg = float(np.rad2deg(shaft_rows[shaft_selected_index, 5]))
shaft_selected_tau = float(shaft_rows[shaft_selected_index, 4])
shaft_selected_ok = bool(shaft_rows[shaft_selected_index, 8])

print("Controle gemeenschappelijke aandrijfas:")
print(f"aslengte tussen mechanismen              : {shaft_span:.2f} m")
print(f"ontwerp askoppel, conservatief           : {shaft_design_torque:.2f} Nm")
print(f"ontwerp koppel per lokale schuiver       : {shaft_torque_per_slider_design:.2f} Nm")
print(f"geschatte radiale lagerlast per poelie   : {shaft_radial_bearing_load_est:.2f} N")
print("optie | Do [mm] | Di [mm] | massa [kg/m] | tau [MPa] | twist [deg] | status")
for i, name in enumerate(shaft_option_names):
    outer_diameter, inner_diameter, _, mass_per_m, tau_max, twist_rad, util_shear, util_twist, ok = shaft_rows[i]
    status = "gekozen" if i == shaft_selected_index else ("haalbaar" if ok else "te slap/zwaar belast")
    print(
        f"{name:9s} | {1000*outer_diameter:7.1f} | {1000*inner_diameter:7.1f} | {mass_per_m:12.2f} | "
        f"{tau_max/1e6:9.2f} | {np.rad2deg(twist_rad):10.3f} | {status}"
    )
print("Interpretatie: een motor is praktisch verdedigbaar als de as torsiestijf genoeg is, de koppelingen dit koppel aankunnen en de pulley-lagers op de riemlast gekozen worden.")

fig_shaft, ax_shaft = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
fig_shaft.suptitle("Gemeenschappelijke aandrijfas")
indices = np.arange(len(shaft_option_names))
shaft_colors = ["tab:green" if ok else "tab:red" for ok in shaft_rows[:, 8].astype(bool)]
shaft_colors[shaft_selected_index] = "tab:blue"

ax_shaft[0].bar(indices, shaft_rows[:, 4] / 1e6, color=shaft_colors)
ax_shaft[0].axhline(shaft_allowable_shear_stress / 1e6, color="tab:red", ls="--", label="limiet")
ax_shaft[0].set_xticks(indices); ax_shaft[0].set_xticklabels(shaft_option_names, rotation=25)
ax_shaft[0].set_ylabel("torsiespanning [MPa]")
ax_shaft[0].set_title("Sterkte")
ax_shaft[0].grid(True, axis="y"); ax_shaft[0].legend()

ax_shaft[1].bar(indices, np.rad2deg(shaft_rows[:, 5]), color=shaft_colors)
ax_shaft[1].axhline(shaft_twist_limit_deg, color="tab:red", ls="--", label="richtlimiet")
ax_shaft[1].set_xticks(indices); ax_shaft[1].set_xticklabels(shaft_option_names, rotation=25)
ax_shaft[1].set_ylabel("verdraaiing [deg]")
ax_shaft[1].set_title("Synchronisatie / torsiestijfheid")
ax_shaft[1].grid(True, axis="y"); ax_shaft[1].legend()

ax_shaft[2].bar(indices, shaft_rows[:, 3], color=shaft_colors)
ax_shaft[2].set_xticks(indices); ax_shaft[2].set_xticklabels(shaft_option_names, rotation=25)
ax_shaft[2].set_ylabel("kg/m")
ax_shaft[2].set_title("Asmassa")
ax_shaft[2].grid(True, axis="y")
plt.show()


## Frequentie-inhoud van de aandrijfbelasting

Deze figuur gebruikt de gekozen motor-loadcase uit Notebook 3. Ze dient om te controleren of het trage traject vooral lage frequenties bevat en dus niet bedoeld is als snel dynamisch mechanisme.



In [ ]:
def one_sided_spectrum(y, dt):
    y = np.asarray(y, dtype=float)
    y = y - np.mean(y)
    if len(y) < 4 or dt <= 0:
        return np.array([0.0]), np.array([0.0])
    window = np.hanning(len(y))
    yw = y * window
    freq = np.fft.rfftfreq(len(yw), d=dt)
    amp = 2.0 * np.abs(np.fft.rfft(yw)) / np.sum(window)
    return freq, amp

freq_acc, amp_acc = one_sided_spectrum(dds_drive, Ts_drive)
freq_force, amp_force = one_sided_spectrum(F_s_drive, Ts_drive)

nonzero_force = freq_force > 0
if np.any(nonzero_force):
    top_idx = np.argsort(amp_force[nonzero_force])[-5:][::-1]
    top_freqs = freq_force[nonzero_force][top_idx]
    top_amps = amp_force[nonzero_force][top_idx]
else:
    top_freqs = np.array([])
    top_amps = np.array([])

drive_reference_freq = float(top_freqs[0]) if len(top_freqs) else (1.0 / T_run if T_run > 0 else np.nan)
beam_modal_line_mass = np.nan
beam_first_bending_freq = np.nan
frequency_separation_ratio = np.nan
if (
    np.isfinite(beam_I_loaded) and beam_I_loaded > 0
    and np.isfinite(beam_span_loaded) and beam_span_loaded > 0
    and np.isfinite(front_beam_mass_per_m_loaded) and front_beam_mass_per_m_loaded > 0
    and np.isfinite(fabric_areal_density_loaded)
    and canopy_depth_loaded is not None and canopy_depth_loaded > 0
    and np.isfinite(aluminium_E_loaded) and aluminium_E_loaded > 0
):
    beam_modal_line_mass = front_beam_mass_per_m_loaded + fabric_areal_density_loaded * canopy_depth_loaded * front_beam_tributary_depth_fraction_loaded
    beam_first_bending_freq = (np.pi**2 / (2.0 * np.pi)) * np.sqrt(aluminium_E_loaded * beam_I_loaded / (beam_modal_line_mass * beam_span_loaded**4))
    frequency_separation_ratio = beam_first_bending_freq / drive_reference_freq if drive_reference_freq > 0 else np.nan

print("Dominante frequenties van de aandrijfkracht:")
for f_i, a_i in zip(top_freqs, top_amps):
    print(f"f = {f_i:.4f} Hz | amplitude = {a_i:.2f} N")
if np.isfinite(beam_first_bending_freq):
    print(f"geschatte eerste buigfrequentie voorbalk : {beam_first_bending_freq:.2f} Hz")
    print(f"scheidingsfactor t.o.v. dominante kracht : {frequency_separation_ratio:.1f}x")
else:
    print("Geen voorbalkdata beschikbaar voor een eigenfrequentie-inschatting.")

fig_freq, ax_freq = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
fig_freq.suptitle(f"Frequentie-inhoud ({motor_design_direction})")

ax_freq[0].plot(freq_acc, amp_acc)
ax_freq[0].set_xlim(0, min(1.0, np.max(freq_acc) if len(freq_acc) else 1.0))
ax_freq[0].set_xlabel("f [Hz]"); ax_freq[0].set_ylabel("amplitude [m/s?]")
ax_freq[0].set_title("Schuiverversnelling")
ax_freq[0].grid(True)

ax_freq[1].plot(freq_force, amp_force)
ax_freq[1].set_xlim(0, min(1.0, np.max(freq_force) if len(freq_force) else 1.0))
ax_freq[1].set_xlabel("f [Hz]"); ax_freq[1].set_ylabel("amplitude [N]")
ax_freq[1].set_title("Aandrijfkracht")
ax_freq[1].grid(True)

if np.isfinite(beam_first_bending_freq) and np.isfinite(drive_reference_freq) and drive_reference_freq > 0:
    ax_freq[2].bar(["aandrijving", "voorbalk f1"], [drive_reference_freq, beam_first_bending_freq], color=["tab:blue", "tab:orange"])
    ax_freq[2].set_yscale("log")
    ax_freq[2].set_ylabel("f [Hz], log")
    ax_freq[2].set_title("Frequentiemarge")
    ax_freq[2].grid(True, axis="y")
else:
    ax_freq[2].text(0.5, 0.5, "geen structurele\nfrequentiereferentie", ha="center", va="center")
    ax_freq[2].set_axis_off()

plt.show()


## Aandrijfschets, kost en energieverbruik

De schets toont het gekozen principe: één motor/reductor met rem op een afgeschermde gemeenschappelijke as boven of achter de constructie. Per mechanisme vertrekt lokaal een riem/kabel naar de schuiver. De as ligt dus niet op wandelhoogte.



In [ ]:
energy_per_open_close_mech = energy_positive_open_one + (energy_positive_close_one if has_bidirectional else 0.0)
energy_per_open_close_total = mechanism_count * energy_per_open_close_mech / drive_efficiency
annual_motion_energy_kwh = energy_per_open_close_total * open_close_cycles_per_day * operating_days_per_year / 3.6e6
annual_standby_energy_kwh = controller_standby_power_w * 24.0 * 365.0 / 1000.0
annual_energy_cost_eur = (annual_motion_energy_kwh + annual_standby_energy_kwh) * electricity_price_eur_per_kwh
installed_drive_cost_min_eur = motor_cost_min_eur + drive_hardware_cost_min_eur
installed_drive_cost_max_eur = motor_cost_max_eur + drive_hardware_cost_max_eur

print("Kost- en energie-inschatting:")
print(f"bewegingsenergie per open+sluit cyclus : {energy_per_open_close_total:.1f} J elektrisch-equivalent")
print(f"jaarlijkse bewegingsenergie            : {annual_motion_energy_kwh:.5f} kWh/jaar")
print(f"jaarlijkse stand-by energie            : {annual_standby_energy_kwh:.2f} kWh/jaar")
print(f"jaarlijkse energiekost                 : {annual_energy_cost_eur:.2f} euro/jaar")
print(f"richtprijs motor + rem/drive           : {motor_cost_min_eur:.0f}-{motor_cost_max_eur:.0f} euro")
print(f"richtprijs riem/as/poelies/lagers      : {drive_hardware_cost_min_eur:.0f}-{drive_hardware_cost_max_eur:.0f} euro")
print(f"totale aandrijf-hardware orde          : {installed_drive_cost_min_eur:.0f}-{installed_drive_cost_max_eur:.0f} euro")

fig_scheme, ax_scheme = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax_scheme.set_title("Praktische aandrijfopstelling")
ax_scheme.set_xlim(-0.5, 6.5); ax_scheme.set_ylim(-0.5, 3.5)
ax_scheme.axis("off")

for x in [0.8, 5.8]:
    ax_scheme.plot([x, x], [0.0, 3.0], color="black", lw=2)
    ax_scheme.add_patch(plt.Rectangle((x - 0.18, 1.45), 0.36, 0.35, fill=False, lw=2))
    ax_scheme.text(x, 1.25, "schuiver", ha="center", va="top")
    ax_scheme.plot([x - 0.15, x - 0.15], [0.45, 2.75], color="tab:blue", lw=2)
    ax_scheme.plot([x + 0.15, x + 0.15], [0.45, 2.75], color="tab:blue", lw=2)
    ax_scheme.text(x + 0.35, 2.6, "lokale riem/kabel", fontsize=9)

ax_scheme.plot([0.8, 5.8], [3.0, 3.0], color="tab:orange", lw=4)
ax_scheme.text(3.3, 3.15, "gemeenschappelijke as boven/achter", ha="center", color="tab:orange")
ax_scheme.add_patch(plt.Circle((0.35, 3.0), 0.22, color="tab:red", alpha=0.8))
ax_scheme.text(0.35, 2.65, "motor\n+ rem", ha="center", va="top")
ax_scheme.annotate("geen as op wandelhoogte", xy=(3.3, 0.4), xytext=(3.3, 1.1), ha="center", arrowprops=dict(arrowstyle="->"))
ax_scheme.plot([0.8, 5.8], [1.7, 1.7], color="0.7", ls="--")
ax_scheme.text(3.3, 1.82, "voorbalk/doek beweegt mee met K-punten", ha="center", fontsize=9)
plt.show()


## Ontwerpconclusie

De motor is niet bedoeld om de schuiver zijdelings te geleiden. De correcte taakverdeling is:

- **motor + reductor + riem/kabel:** verticale aandrijfkracht en positionering;
- **rem/vergrendeling:** veilig vasthouden in open, gesloten en tussenstanden;
- **schuiver/collar + mastgeleiding:** horizontale steunreacties en kantelmomenten;
- **controller + encoder + eindschakelaars:** precisie, homing en eindpositiebeveiliging.

Een vliegwiel is hier niet de eerste logische keuze, omdat de overdekking een traag positioneermechanisme met stilstand in willekeurige tussenstanden is. Voor energie-opslag of zwaartekrachtcompensatie is een trekveer of gasveer logischer. De single-mechanism trekveer-case komt uit `Notebook 3 - Trekveren.ipynb` met `load_case = "trekveren"`. De brede overdekking komt uit `Notebook 3 - Overdekking.ipynb` met `load_case = "overdekking"` of `load_case = "overdekking_trekveren"`.

Voor de overdekkingscase betekent `mechanism_count` het aantal schuivers dat door dezelfde motor/as wordt aangedreven. De motor levert dan het totaal van de verticale riemkrachten; elke lokale schuivergeleiding blijft apart verantwoordelijk voor de zijreacties.



In [ ]:
if load_case == "trekveren":
    results4_filename = "notebook4_aandrijving_trekveren_results.npz"
elif load_case == "overdekking":
    results4_filename = "notebook4_aandrijving_overdekking_results.npz"
elif load_case == "overdekking_trekveren":
    results4_filename = "notebook4_aandrijving_overdekking_trekveren_results.npz"
elif load_case == "custom":
    results4_filename = "notebook4_aandrijving_custom_results.npz"
else:
    results4_filename = "notebook4_aandrijving_results.npz"
results4_path = Path(results4_filename).resolve()
np.savez(
    results4_path,
    t=t, s=s, ds=ds, dds=dds,
    load_case=np.array(load_case),
    source_results3_filename=np.array(str(results3_path.name)),
    spring_case=("spring_count" in data.files),
    drive_type=np.array(drive_type),
    mechanism_count=mechanism_count,
    mechanism_count_override=np.array("None" if mechanism_count_override is None else str(mechanism_count_override)),
    loadcase_mechanism_count=-1 if loadcase_mechanism_count is None else loadcase_mechanism_count,
    has_bidirectional=has_bidirectional,
    direction_names=direction_names,
    motor_design_direction=np.array(motor_design_direction),
    power_design_direction=np.array(power_design_direction),
    direction_force_peaks_one=direction_force_peaks_one,
    direction_power_peaks_one=direction_power_peaks_one,
    direction_positive_energy_one=direction_positive_energy_one,
    direction_net_energy_one=direction_net_energy_one,
    t_drive=t_drive,
    s_drive_profile=s_drive_profile,
    ds_drive=ds_drive,
    dds_drive=dds_drive,
    auto_select_pulley=auto_select_pulley,
    preferred_pulley_radius=preferred_pulley_radius,
    drive_pulley_radius=drive_pulley_radius,
    pulley_radius_feasible=pulley_radius_feasible,
    pulley_selection_note=np.array(pulley_selection_note),
    max_preferred_drive_torque=max_preferred_drive_torque,
    max_preferred_brake_torque=max_preferred_brake_torque,
    drive_efficiency=drive_efficiency,
    gear_efficiency=gear_efficiency,
    selected_gear_ratio=selected_gear_ratio,
    gear_selection_note=np.array(gear_selection_note),
    motor_speed_peak_est=motor_speed_peak_est,
    motor_speed_avg_est=motor_speed_avg_est,
    motor_torque_design_est=motor_torque_design_est,
    drive_safety_factor=drive_safety_factor,
    brake_safety_factor=brake_safety_factor,
    line_force_peak_motion=line_force_peak_motion,
    line_force_peak_motion_open=line_force_peak_motion_open,
    line_force_peak_motion_close=line_force_peak_motion_close,
    line_force_peak_hold=line_force_peak_hold,
    line_force_peak_operating=line_force_peak_operating,
    line_force_design=line_force_design,
    T_pulley_active=T_pulley_active,
    T_pulley_peak_active=T_pulley_peak_active,
    T_pulley_design=T_pulley_design,
    P_slider_drive=P_slider_drive,
    P_slider_drive_open=P_slider_drive_open,
    P_slider_drive_close=np.array([]) if P_slider_drive_close is None else P_slider_drive_close,
    P_peak_slider=P_peak_slider,
    P_peak_slider_open=P_peak_slider_open,
    P_peak_slider_close=P_peak_slider_close,
    P_rms_slider=P_rms_slider,
    P_peak_motor_input=P_peak_motor_input,
    P_rms_motor_input=P_rms_motor_input,
    I_peak_est=I_peak_est,
    I_rms_est=I_rms_est,
    E_positive_slider=E_positive_slider,
    E_positive_slider_open=E_positive_slider_open,
    E_positive_slider_close=E_positive_slider_close,
    E_net_slider=E_net_slider,
    E_net_slider_open=E_net_slider_open,
    E_net_slider_close=E_net_slider_close,
    A_theta_drive=A_theta_drive,
    A_max_drive=A_max_drive,
    pulley_speed_rpm=pulley_speed_rpm,
    avg_equiv_output_rpm=avg_equiv_output_rpm,
    peak_output_rpm=peak_output_rpm,
    hold_s_curve=hold_s_curve,
    F_hold_drive_curve=F_hold_drive_curve,
    T_hold_output_curve=T_hold_output_curve,
    T_hold_brake_design_curve=T_hold_brake_design_curve,
    T_hold_motor_shaft_curve=T_hold_motor_shaft_curve,
    side_reaction_peak=side_reaction_peak,
    local_bending_est=local_bending_est,
    radius_options=pulley_radius_candidates,
    radius_rows=np.array(radius_rows, dtype=float),
    gear_ratio_candidates=gear_ratio_candidates,
    gear_rows=np.array(gear_rows, dtype=float),
    time_scale_factors=time_scale_factors,
    scale_rows=np.array(scale_rows, dtype=float),
    comparison_rows=np.array(comparison_rows, dtype=float),
    shaft_span=shaft_span,
    shaft_design_torque=shaft_design_torque,
    shaft_peak_active_torque=shaft_peak_active_torque,
    shaft_torque_per_slider_design=shaft_torque_per_slider_design,
    shaft_option_names=shaft_option_names,
    shaft_options=np.array(shaft_options, dtype=object),
    shaft_rows=shaft_rows,
    shaft_selected_name=np.array(shaft_selected_name),
    shaft_selected_diameter=shaft_selected_diameter,
    shaft_selected_inner_diameter=shaft_selected_inner_diameter,
    shaft_selected_mass_per_m=shaft_selected_mass_per_m,
    shaft_selected_twist_deg=shaft_selected_twist_deg,
    shaft_selected_tau=shaft_selected_tau,
    shaft_selected_ok=shaft_selected_ok,
    shaft_material_G=shaft_material_G,
    shaft_density=shaft_density,
    shaft_allowable_shear_stress=shaft_allowable_shear_stress,
    shaft_twist_limit_deg=shaft_twist_limit_deg,
    belt_bearing_load_factor=belt_bearing_load_factor,
    local_line_force_design=local_line_force_design,
    shaft_radial_bearing_load_est=shaft_radial_bearing_load_est,
    shaft_coupling_torque_design=shaft_coupling_torque_design,
    freq_acc=freq_acc,
    amp_acc=amp_acc,
    freq_force=freq_force,
    amp_force=amp_force,
    top_force_freqs=top_freqs,
    top_force_amps=top_amps,
    drive_reference_freq=drive_reference_freq,
    beam_modal_line_mass=beam_modal_line_mass,
    beam_first_bending_freq=beam_first_bending_freq,
    frequency_separation_ratio=frequency_separation_ratio,
    linear_resolution_motor_encoder=linear_resolution_motor_encoder,
    linear_resolution_output_encoder=linear_resolution_output_encoder,
    elastic_deflection_peak=elastic_deflection_peak,
    elastic_deflection_design=elastic_deflection_design,
    energy_per_open_close_total=energy_per_open_close_total,
    annual_motion_energy_kwh=annual_motion_energy_kwh,
    annual_standby_energy_kwh=annual_standby_energy_kwh,
    annual_energy_cost_eur=annual_energy_cost_eur,
    installed_drive_cost_min_eur=installed_drive_cost_min_eur,
    installed_drive_cost_max_eur=installed_drive_cost_max_eur,
    motor_class_names=motor_class_names,
    motor_class_power_w=motor_class_power_w,
    motor_class_output_torque_nm=motor_class_output_torque_nm,
    motor_class_brake_torque_nm=motor_class_brake_torque_nm,
    motor_class_cost_min_eur=motor_class_cost_min_eur,
    motor_class_cost_max_eur=motor_class_cost_max_eur,
    motor_class_has_encoder=motor_class_has_encoder,
    motor_class_has_brake=motor_class_has_brake,
    motor_class_suitable=motor_class_suitable,
    selected_motor_class_name=np.array(selected_motor_class_name),
    selected_motor_power_class=selected_motor_power_class,
    selected_output_torque_class=selected_output_torque_class,
    selected_brake_torque_class=selected_brake_torque_class,
    selected_motor_cost_min=selected_motor_cost_min,
    selected_motor_cost_max=selected_motor_cost_max,
    selected_motor_ip_rating=np.array(selected_motor_ip_rating),
    motor_power_margin=motor_power_margin,
    motor_torque_margin=motor_torque_margin,
    motor_brake_margin=motor_brake_margin,
    motor_cost_min_eur=motor_cost_min_eur,
    motor_cost_max_eur=motor_cost_max_eur,
    drive_hardware_cost_min_eur=drive_hardware_cost_min_eur,
    drive_hardware_cost_max_eur=drive_hardware_cost_max_eur,
    recommended_motor_power_peak=recommended_motor_power_peak,
    recommended_output_torque=recommended_output_torque,
    recommended_brake_torque=recommended_brake_torque,
    canopy_width=np.nan if canopy_width_loaded is None else canopy_width_loaded,
    canopy_depth=np.nan if canopy_depth_loaded is None else canopy_depth_loaded,
    payload_mass_K_equivalent=np.nan if payload_mass_K_loaded is None else payload_mass_K_loaded,
)

print("Notebook 4-resultaten opgeslagen in:")
print(results4_path)
print()
print("SAMENVATTING - AANDRIJVING")
print("=" * 56)
print(f"load_case                         : {load_case}")
print(f"bronbestand                       : {results3_path.name}")
print(f"voorkeursconcept                  : 24/48 V DC/BLDC reductiemotor + encoder + rem")
print(f"aandrijving                       : gemeenschappelijke as + lokale tandriem/kabel per schuiver")
print(f"maatgevende richting kracht       : {motor_design_direction}")
print(f"maatgevende richting vermogen     : {power_design_direction}")
print(f"gekozen poelieradius              : {drive_pulley_radius*1000:.1f} mm")
print(f"gekozen reductie                  : {selected_gear_ratio:.0f}:1")
print(f"piek lijnkracht per mechanisme    : {float(np.nanmax(direction_force_peaks_one)):.2f} N")
print(f"piek lijnkracht, totaal operationeel: {line_force_peak_operating:.2f} N")
print(f"gekozen ontwerplijnkracht          : {line_force_design:.2f} N")
print(f"ontwerp aandrijfkoppel incl. verlies: {T_pulley_design:.2f} Nm")
print(f"max remkoppel aan uitgang          : {np.max(T_hold_brake_design_curve):.2f} Nm")
print(f"max remkoppel aan motoras          : {np.max(T_hold_motor_shaft_curve):.3f} Nm")
print(f"piek motor-ingangsvermogen ontwerp : {P_peak_motor_input:.2f} W")
print(f"aanbevolen motorvermogen praktisch : {recommended_motor_power_peak:.1f} W klasse")
print(f"aanbevolen uitgangskoppel praktisch: {recommended_output_torque:.1f} Nm klasse")
print(f"gekozen motorclass                  : {selected_motor_class_name}")
print(f"motorclass marges P/T/rem           : {motor_power_margin:.2f} / {motor_torque_margin:.2f} / {motor_brake_margin:.2f}")
print(f"vereist piek-uitgangstoerental     : {peak_output_rpm:.1f} rpm")
print(f"gemiddeld equivalent toerental     : {avg_equiv_output_rpm:.1f} rpm")
print(f"geschatte motorpieksnelheid        : {motor_speed_peak_est:.0f} rpm")
print(f"max lokale zijreactie schuiver      : {side_reaction_peak:.2f} N")
print(f"gekozen asoptie                     : {shaft_selected_name}, twist {shaft_selected_twist_deg:.2f} deg")
print(f"geschatte pulley-lagerlast          : {shaft_radial_bearing_load_est:.2f} N per lokale poelie")
print(f"arbeids-surplus aandrijving         : {A_max_drive:.2f} J")
print(f"energie per open+sluit cyclus       : {energy_per_open_close_total:.1f} J")
print(f"jaarlijkse energiekost              : {annual_energy_cost_eur:.2f} euro/jaar")
print(f"richtprijs aandrijfhardware       : {installed_drive_cost_min_eur:.0f}-{installed_drive_cost_max_eur:.0f} euro")
print()
print("Kernadvies:")
print("Gebruik een motor met encoder en rem. Laat de riem alleen de verticale aandrijfkracht leveren.")
print("De gemeenschappelijke as hoort boven/achter de constructie; voetgangers stappen nergens over.")
print("Dimensioneer de schuivergeleiding apart op de grote zijreactie; die hoort niet in de riem.")
